# E4 — Malla 3D → STL imprimible

Toma la malla generada por E3 (`.stl`, `.obj` o `.ply`) y produce un STL watertight listo para imprimir.

```
[malla E3]  →  cargar  →  reparar  →  escalar  →  validar  →  [STL + report.json]
```

**Reparaciones aplicadas:**
1. Hacer manifold (manifold3d — algoritmo ManifoldPlus)
2. Rellenar huecos residuales (trimesh)
3. Fijar normales hacia fuera (trimesh)
4. Eliminar caras degeneradas
5. Conservar componente principal
6. Escalar a tamaño de impresión

**Validación de salida:** watertight · manifold · volumen positivo · euler_number=2

---
## Sección 1 — Setup

In [ ]:
from google.colab import drive
import os
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print('Drive ya montado.')
else:
    drive.mount('/content/drive')
    print('Drive montado.')

In [ ]:
import subprocess

for pkg in ['trimesh', 'manifold3d', 'pymeshlab', 'plotly', 'numpy']:
    r = subprocess.run(['pip', 'install', pkg, '-q'], capture_output=True, text=True)
    print(f'[{"OK" if r.returncode==0 else "WARN"}] {pkg}')

# Clonar repo TFM para acceder a los STLs generados por E3
from getpass import getpass
REPO_DIR = '/content/TFM'
if not os.path.exists(REPO_DIR):
    token = getpass('Token GitHub (ghp_...): ')
    subprocess.run(['git','clone',f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D',
                    REPO_DIR, '-q'], capture_output=True)
    del token; print('Repo clonado.')
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','-q'], capture_output=True)
    print('[OK] Repo actualizado.')

os.chdir(REPO_DIR)
subprocess.run(['git','checkout','raquel/e3','-q'], capture_output=True)

---
## Sección 2 — Configuración

Elige el STL de entrada y el tamaño de impresión.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURACION — solo cambia esta sección
# ══════════════════════════════════════════════════════════════

VERSION_E3  = 'v6_obj_sn'   # modelo PoinTr a usar
TAMANO_MM   = 100.0          # lado mayor del bbox en mm para impresión
N_MUESTRAS  = 4              # muestras del test set si no hay STLs en Drive
POISSON_DEPTH = 9            # profundidad Poisson (8=suave, 9=normal, 10=detalle)

# STL de entrada manual (None = busca en Drive o genera automáticamente)
RUTA_ENTRADA = None

# Rutas
DRIVE    = '/content/drive/MyDrive'
BASE_E3  = f'{DRIVE}/Datos_E2_E3/E3/Raquel'
BASE_GEN = f'{DRIVE}/Datos_E2_E3/General'
BASE_E4  = f'{DRIVE}/E4/Raquel'

ENTRADA_DIR  = f'{BASE_E3}/stl/{VERSION_E3}'
SALIDA_DIR   = f'E4/stl_reparados/{VERSION_E3}'
SALIDA_DRIVE = f'{BASE_E4}/stl_reparados/{VERSION_E3}'

_VERSION_DATASETS = {
    'v5_obj':    ['obj'], 'v5_fb_obj': ['fb','obj'],
    'v6_obj_sn': ['obj','sn'], 'v6_all': ['fb','obj','sn'],
}
_partes = _VERSION_DATASETS.get(VERSION_E3, ['obj'])

import os
from pathlib import Path
Path(SALIDA_DIR).mkdir(parents=True, exist_ok=True)

# Detectar STLs disponibles
if RUTA_ENTRADA:
    stls_entrada = [Path(RUTA_ENTRADA)]
    print(f'Entrada manual: {stls_entrada[0].name}')
else:
    stls_entrada = sorted(Path(ENTRADA_DIR).glob('*.stl')) if Path(ENTRADA_DIR).exists() else []
    if stls_entrada:
        print(f'STLs en Drive ({len(stls_entrada)}):')
        for s in stls_entrada: print(f'  {s.name}  ({s.stat().st_size/1024:.0f} KB)')
    else:
        print(f'[!] No hay STLs en {ENTRADA_DIR}')
        print(f'    --> Seccion 0 los generara desde PoinTr {VERSION_E3}')

print(f'\nTamano objetivo: {TAMANO_MM} mm | Poisson depth: {POISSON_DEPTH}')
print(f'Salida Drive: {SALIDA_DRIVE}')

---
## Sección 3 — Cargar y diagnosticar

Inspecciona el estado inicial de cada malla antes de reparar.

In [ ]:
# ── SECCIÓN 0: Generar STLs desde cero si no hay ninguno ───────
# Se salta automáticamente si stls_entrada ya tiene archivos.

if stls_entrada:
    print(f'[OK] Ya hay {len(stls_entrada)} STL(s) — saltando generación.')
else:
    print('Generando STLs desde PoinTr...')
    import sys, types, glob as _glob, random, subprocess, importlib
    import torch, torch.nn as nn, numpy as np
    from pathlib import Path

    # Instalar easydict si falta
    subprocess.run(['pip', 'install', 'easydict', '-q'], capture_output=True)
    from easydict import EasyDict

    # Clonar PoinTr si no existe (necesario para importar models.*)
    if not os.path.exists('/content/PoinTr'):
        subprocess.run(['git', 'clone', 'https://github.com/yuxumin/PoinTr',
                        '/content/PoinTr', '-q'], capture_output=True)
        print('PoinTr clonado.')
    else:
        print('[OK] PoinTr ya existe.')

    # Añadir rutas: PoinTr al índice 0 para que models.* se encuentre primero
    for p in ['/content/TFM', '/content/PoinTr']:
        if p in sys.path: sys.path.remove(p)
    sys.path.insert(0, '/content/TFM')
    sys.path.insert(0, '/content/PoinTr')   # queda en índice 0
    importlib.invalidate_caches()            # limpia caché negativo de importaciones

    os.chdir('/content/TFM')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    DEVICE_STR = str(device)

    # ── Mocks CUDA ───────────────────────────────────────────
    _pfx = ('models','utils.registry','utils.config','utils.logger','utils.misc','extensions')
    _del = [k for k,v in sys.modules.items()
            if any(k==p or k.startswith(p+'.') for p in _pfx)
            or (hasattr(v,'__file__') and v.__file__ and '/content/PoinTr' in str(v.__file__))]
    for k in _del: del sys.modules[k]
    importlib.invalidate_caches()  # segunda pasada tras limpiar sys.modules

    for fp in _glob.glob('/content/PoinTr/models/*.py')+_glob.glob('/content/PoinTr/models/**/*.py'):
        try:
            src=open(fp,encoding='utf-8').read(); new=src.replace('.cuda()',f'.to("{DEVICE_STR}")')
            if new!=src: open(fp,'w',encoding='utf-8').write(new)
        except: pass

    def _force(n,a):
        m=types.ModuleType(n)
        for k,v in a.items(): setattr(m,k,v)
        sys.modules[n]=m
    def _inject(n,a):
        if n not in sys.modules: _force(n,a)
    def _cr(a,b): d=torch.cdist(a,b,p=2); return d.min(2).values,d.min(1).values
    class _CL1(nn.Module):
        def forward(self,a,b): d1,d2=_cr(a.contiguous(),b.contiguous()); return (d1.mean()+d2.mean())/2
    _ch={'ChamferDistanceL1':_CL1,'ChamferDistanceL2':_CL1,'ChamferDistanceL1_PM':_CL1,'chamfer_3DDist':_cr}
    for n in ['chamfer','chamfer_dist','extensions.chamfer_dist','chamfer3D','chamfer3D.dist_chamfer_3D']: _force(n,_ch)
    if 'pointnet2_ops' not in sys.modules:
        def _fps(xyz,np_):
            B,N,_=xyz.shape; dev=xyz.device
            idx=torch.zeros(B,np_,dtype=torch.int32,device=dev); dist=torch.full((B,N),1e10,device=dev)
            far=torch.randint(0,N,(B,),dtype=torch.long,device=dev); bi=torch.arange(B,dtype=torch.long,device=dev)
            for i in range(np_):
                idx[:,i]=far.int(); c=xyz[bi,far].unsqueeze(1)
                dist=torch.min(dist,((xyz-c)**2).sum(-1)); far=dist.max(-1)[1]
            return idx
        def _go(f,idx): B,C,N=f.shape; M=idx.shape[1]; return f.gather(2,idx.long().unsqueeze(1).expand(B,C,M)).contiguous()
        def _bq(r,ns,xyz,nxyz):
            d=torch.cdist(nxyz.float(),xyz.float()); s=d.argsort(-1)[:,:,:ns]
            return torch.where(d.gather(2,s)>r,s[:,:,:1].expand_as(s),s).int()
        def _grp(f,idx):
            B,C,N=f.shape; S,K=idx.shape[1],idx.shape[2]
            return f.gather(2,idx.long().view(B,1,S*K).expand(B,C,S*K)).view(B,C,S,K).contiguous()
        def _3nn(u,k): d=torch.cdist(u.float(),k.float()); d2,i=d.topk(3,-1,largest=False); return d2.float(),i.int()
        def _3i(f,idx,w):
            B,C,M=f.shape; N=idx.shape[1]
            return (f.gather(2,idx.long().view(B,1,N*3).expand(B,C,N*3)).view(B,C,N,3)*w.unsqueeze(1)).sum(-1).contiguous()
        _pu=types.ModuleType('pointnet2_ops.pointnet2_utils')
        for k,v in {'furthest_point_sample':_fps,'gather_operation':_go,'ball_query':_bq,
                    'grouping_operation':_grp,'three_nn':_3nn,'three_interpolate':_3i}.items(): setattr(_pu,k,v)
        _pm=types.ModuleType('pointnet2_ops'); _pm.pointnet2_utils=_pu
        sys.modules['pointnet2_ops']=_pm; sys.modules['pointnet2_ops.pointnet2_utils']=_pu
    class _KNN(nn.Module):
        def __init__(self,k,transpose_mode=False): super().__init__(); self.k=k; self.tm=transpose_mode
        def forward(self,ref,query):
            if self.tm: d=torch.cdist(query.float(),ref.float()); dk,ik=d.topk(self.k,-1,largest=False); return dk,ik
            r=ref.transpose(1,2).contiguous(); q=query.transpose(1,2).contiguous()
            d=torch.cdist(q.float(),r.float()); dk,ik=d.topk(self.k,-1,largest=False)
            return dk.transpose(1,2),ik.transpose(1,2)
    _force('knn_cuda',{'KNN':_KNN})
    def _dc(n): return type(n,(nn.Module,),{'__init__':lambda s,*a,**k:super(type(s),s).__init__(),'forward':lambda s,x,*a,**k:x})
    class _Emd(nn.Module):
        def forward(self,a,b): return torch.zeros(a.shape[0],device=a.device),torch.zeros(a.shape[0],dtype=torch.int32,device=a.device)
    for base,attrs in [('gridding',{'Gridding':_dc('G'),'GriddingReverse':_dc('GR')}),
                       ('gridding_loss',{'GriddingLoss':_dc('GL')}),
                       ('cubic_feature_sampling',{'CubicFeatureSampling':_dc('CFS')}),
                       ('emd',{'emd_module':_Emd,'EarthMoverDistance':_Emd})]:
        for pfx in ['','extensions.']: _inject(pfx+base,attrs)

    # Verificar que sys.path tiene PoinTr antes de importar
    print(f'sys.path[0:3] = {sys.path[:3]}')

    # ── Cargar modelo ────────────────────────────────────────
    ckpt_path = f'E3/checkpoints_pointr_{VERSION_E3}/best.pt'
    if not Path(ckpt_path).exists():
        ckpt_path = f'{BASE_E3}/modelos/{VERSION_E3}/best.pt'
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    print(f'Modelo PoinTr {VERSION_E3} — epoch {ck["epoch"]}')
    try:
        from models.build import build_model_from_cfg; model=build_model_from_cfg(EasyDict(ck['model_cfg']))
    except:
        from models.PoinTr import PoinTr; model=PoinTr(EasyDict(ck['model_cfg']))
    model.load_state_dict(ck['model_state_dict']); model=model.to(device).eval()

    # ── Dataset → test set ───────────────────────────────────
    import E3.dataset as _ds; _ds.CENTRAR_EN_ROTO=False
    from E3.dataset import construir_pares
    FUENTES = {'obj':(f'{BASE_GEN}/roturas_Objaverse_v2','Datos/objaverse/roturas_v2'),
               'sn': (f'{BASE_GEN}/shapenet_roturas','Datos/shapenet/roturas'),
               'fb': (f'{BASE_GEN}/Fantastik_Break_Procesado_v2','Datos/fantastic_breaks/procesado_v2')}
    carpetas = [v[1] for k,v in FUENTES.items() if k in _partes and Path(v[1]).exists()]
    if not carpetas:
        carpetas = [v[0] for k,v in FUENTES.items() if k in _partes and Path(v[0]).exists()]
    todos = construir_pares(carpetas)
    rng = random.Random(42); rng.shuffle(todos)
    n = len(todos); nt=int(0.8*n); nv=int(0.1*n)
    test = todos[nt+nv:]
    muestra = random.sample(test, min(N_MUESTRAS, len(test)))
    print(f'Test set: {len(test)} pares → mostrando {len(muestra)}')

    # ── Inferencia + Poisson ──────────────────────────────────
    import pymeshlab
    stl_dir = Path(f'E3/stl_generados/{VERSION_E3}'); stl_dir.mkdir(parents=True, exist_ok=True)

    for ruta_r, _ in muestra:
        roto = np.load(ruta_r).astype(np.float32)
        with torch.no_grad():
            inp = torch.tensor(roto).unsqueeze(0).to(device)
            out = model(inp)
            pred = (out[-1] if isinstance(out,(list,tuple)) else out).squeeze(0).cpu().numpy()
        nombre = Path(ruta_r).stem.replace('_roto','')
        print(f'  Inferencia OK: {nombre}')

        ms = pymeshlab.MeshSet()
        ms.add_mesh(pymeshlab.Mesh(vertex_matrix=pred.astype(np.float64)), nombre)
        ms.compute_normal_for_point_clouds(k=30, smoothiter=2)
        ms.generate_surface_reconstruction_screened_poisson(depth=POISSON_DEPTH, scale=1.1)
        ms.meshing_remove_connected_component_by_face_number(mincomponentsize=500)
        ms.apply_coord_laplacian_smoothing(stepsmoothnum=2)

        ruta_stl = stl_dir / f'{nombre}.stl'
        ms.save_current_mesh(str(ruta_stl))
        print(f'    STL: {ruta_stl.name}  ({ruta_stl.stat().st_size/1024:.0f} KB)')

    stls_entrada = sorted(stl_dir.glob('*.stl'))
    print(f'\n{len(stls_entrada)} STLs generados → continuando con reparacion.')

---
## Sección 0 — Generar STLs desde PoinTr (solo si no hay STLs en Drive)

**Salta esta sección si ya tienes STLs en Drive.** Si `stls_entrada` está vacío, esta sección carga el modelo PoinTr, toma muestras del test set, corre inferencia y genera las mallas con Poisson (pymeshlab).

In [ ]:
import trimesh
import numpy as np
from pathlib import Path

def diagnosticar(mesh, etiqueta=''):
    wt = mesh.is_watertight
    vol = mesh.volume if wt else None
    eu  = mesh.euler_number
    bb  = mesh.bounding_box.extents
    print(f'  {etiqueta}')
    print(f'    Vertices: {len(mesh.vertices):>6}   Caras: {len(mesh.faces):>6}')
    print(f'    Watertight : {"SI" if wt else "NO"}')
    print(f'    Euler num  : {eu}  (esperado: 2 para impresion)')
    print(f'    Volumen    : {f"{vol*1000:.2f} cm3" if vol else "N/A (no watertight)"}')
    print(f'    BBox       : {bb[0]:.3f} x {bb[1]:.3f} x {bb[2]:.3f} (unidades originales)')
    return {'watertight': wt, 'euler': eu, 'volumen': vol, 'bbox': bb.tolist(),
            'vertices': len(mesh.vertices), 'caras': len(mesh.faces)}

mallas_cargadas = []

for ruta in stls_entrada:
    print(f'\n── {ruta.name} ──')
    try:
        mesh = trimesh.load(str(ruta), force='mesh')
        if isinstance(mesh, trimesh.Scene):
            mesh = trimesh.util.concatenate(list(mesh.geometry.values()))
        stats_ini = diagnosticar(mesh, 'ANTES de reparar')
        mallas_cargadas.append({'nombre': ruta.stem, 'ruta': str(ruta),
                                 'mesh': mesh, 'stats_ini': stats_ini})
    except Exception as e:
        print(f'  [ERROR] {e}')

print(f'\n{len(mallas_cargadas)} mallas cargadas. Pasa a Seccion 4 para reparar.')

---
## Sección 4 — Reparar

Pipeline de reparación en 4 pasos:

1. **manifold3d** — convierte a manifold (ManifoldPlus): resuelve aristas no-manifold y caras fantasma
2. **fill_holes** — rellena huecos residuales
3. **fix_normals** — orientación coherente hacia fuera
4. **componente principal** — elimina fragmentos sueltos

In [ ]:
import trimesh
import numpy as np
from pathlib import Path

# Intentar manifold3d (ManifoldPlus)
try:
    import manifold3d
    TIENE_MANIFOLD = True
    print('[OK] manifold3d disponible')
except ImportError:
    TIENE_MANIFOLD = False
    print('[WARN] manifold3d no disponible — usando solo trimesh')

def hacer_manifold(mesh):
    if not TIENE_MANIFOLD:
        return mesh, False
    try:
        m = manifold3d.Manifold(
            manifold3d.Mesh(
                vert_properties=np.array(mesh.vertices, dtype=np.float32),
                tri_verts=np.array(mesh.faces, dtype=np.uint32)
            )
        )
        out = m.to_mesh()
        resultado = trimesh.Trimesh(
            vertices=np.array(out.vert_properties),
            faces=np.array(out.tri_verts),
            process=False
        )
        if len(resultado.vertices) == 0:
            print('  [WARN manifold3d] devolvió malla vacía — ignorando')
            return mesh, False
        return resultado, True
    except Exception as e:
        print(f'  [WARN manifold3d] {e} — continuando sin manifold')
        return mesh, False

def reparar(mesh):
    reps = []

    # Guard inicial
    if len(mesh.vertices) == 0 or len(mesh.faces) == 0:
        return mesh, ['malla_vacia_entrada']

    # 1. Componente principal (antes de manifold para no procesar basura)
    try:
        componentes = mesh.split(only_watertight=False)
        if len(componentes) > 1:
            mejor = max(componentes, key=lambda c: len(c.faces))
            if len(mejor.faces) > 0:
                mesh = mejor
                reps.append(f'componente_principal ({len(componentes)} → 1)')
    except Exception as e:
        print(f'  [WARN split] {e}')

    # 2. ManifoldPlus
    mesh, ok = hacer_manifold(mesh)
    if ok:
        reps.append('manifold3d')

    # 3. Rellenar huecos
    n_antes = len(mesh.faces)
    trimesh.repair.fill_holes(mesh)
    if len(mesh.faces) != n_antes:
        reps.append(f'fill_holes (+{len(mesh.faces)-n_antes} caras)')

    # 4. Fijar normales
    trimesh.repair.fix_normals(mesh)
    trimesh.repair.fix_winding(mesh)
    reps.append('fix_normals')

    # 5. Caras degeneradas
    mask = mesh.nondegenerate_faces()
    n_deg = (~mask).sum()
    if n_deg > 0:
        mesh.update_faces(mask)
        reps.append(f'remove_degenerate ({n_deg} caras)')

    # 6. Procesar (limpia vértices no referenciados, etc.)
    mesh.process(validate=False)

    return mesh, reps

mallas_reparadas = []

for r in mallas_cargadas:
    nombre = r['nombre']
    print(f'\n── {nombre} ──')
    mesh_rep, reps = reparar(r['mesh'])
    print(f'  Reparaciones: {" | ".join(reps)}')

    if len(mesh_rep.vertices) == 0:
        print(f'  [WARN] malla vacía tras reparar — se omite en exportación')

    wt = mesh_rep.is_watertight
    eu = mesh_rep.euler_number
    print(f'  Resultado: watertight={"SI" if wt else "NO"}  euler={eu}  '
          f'vertices={len(mesh_rep.vertices)}  caras={len(mesh_rep.faces)}')

    r['mesh_rep'] = mesh_rep
    r['reparaciones'] = reps
    mallas_reparadas.append(r)

print(f'\n{len(mallas_reparadas)} mallas reparadas.')

---
## Sección 5 — Escalar a tamaño de impresión

El modelo E3 trabaja en unidades de esfera unitaria (radio = 1). Aquí se convierte a mm reales para que la impresora entienda el tamaño.

In [ ]:
import numpy as np

mallas_escaladas = []

for r in mallas_reparadas:
    mesh = r['mesh_rep']
    nombre = r['nombre']

    # Guard: malla vacía tras la reparación
    if len(mesh.vertices) == 0 or len(mesh.faces) == 0:
        print(f'[WARN] {nombre}: malla vacía tras reparar — omitida')
        continue
    if mesh.bounds is None:
        print(f'[WARN] {nombre}: bounds=None (malla degenerada) — omitida')
        continue

    # Centrar en origen
    mesh.apply_translation(-mesh.centroid)

    # Escalar: lado mayor del bbox → TAMANO_MM
    lado_max = mesh.bounding_box.extents.max()
    if lado_max > 0:
        factor = TAMANO_MM / lado_max
        mesh.apply_scale(factor)
    else:
        print(f'[WARN] {nombre}: bbox nulo — sin escalar')
        factor = 1.0

    bb = mesh.bounding_box.extents
    r['mesh_final'] = mesh
    r['escala_mm'] = factor
    r['bbox_mm'] = bb.tolist()
    mallas_escaladas.append(r)

    print(f'{nombre}: x{factor:.1f}  →  {bb[0]:.1f} x {bb[1]:.1f} x {bb[2]:.1f} mm')

# Usar mallas_escaladas en secciones siguientes
mallas_reparadas = mallas_escaladas
print(f'\n{len(mallas_reparadas)} mallas listas para exportar.')

---
## Sección 6 — Validar y exportar

Comprueba los requisitos de impresión y guarda el STL + `report.json`.

In [ ]:
import json, shutil
from pathlib import Path
from datetime import date

CRITERIOS = {
    'watertight':   'Cerrada sin huecos (obligatorio para impresión)',
    'euler_ok':     'Euler number = 2 (género 0, sólido simple)',
    'volumen_ok':   'Volumen positivo',
    'min_faces':    'Al menos 100 caras',
}

informes = []

for r in mallas_reparadas:
    nombre  = r['nombre']
    mesh    = r['mesh_final']
    print(f'\n── {nombre} ──')

    wt  = mesh.is_watertight
    eu  = mesh.euler_number
    vol = mesh.volume if wt else None
    nf  = len(mesh.faces)

    checks = {
        'watertight': wt,
        'euler_ok':   eu == 2,
        'volumen_ok': vol is not None and vol > 0,
        'min_faces':  nf >= 100,
    }
    apto = all(checks.values())

    for k, v in checks.items():
        print(f'  {"OK" if v else "!!"}  {CRITERIOS[k]}')
    print(f'  --> APTO PARA IMPRIMIR: {"SI" if apto else "NO — requiere revision manual"}')
    if vol:
        print(f'  Volumen: {vol/1000:.2f} cm3  |  BBox: {r["bbox_mm"][0]:.1f}x{r["bbox_mm"][1]:.1f}x{r["bbox_mm"][2]:.1f} mm')

    # Guardar STL
    ruta_stl = Path(SALIDA_DIR) / f'{nombre}_E4.stl'
    mesh.export(str(ruta_stl))
    tam_kb = ruta_stl.stat().st_size / 1024
    print(f'  STL: {ruta_stl.name}  ({tam_kb:.0f} KB)')

    # report.json
    informe = {
        'nombre': nombre,
        'archivo_entrada': r['ruta'],
        'archivo_salida': str(ruta_stl),
        'fecha': str(date.today()),
        'modelo_e3': VERSION_E3,
        'apto_para_imprimir': apto,
        'watertight': wt,
        'euler_number': eu,
        'volumen_cm3': round(vol/1000, 3) if vol else None,
        'n_vertices': len(mesh.vertices),
        'n_caras': nf,
        'bbox_mm': [round(x, 2) for x in r['bbox_mm']],
        'tamano_objetivo_mm': TAMANO_MM,
        'reparaciones': r['reparaciones'],
        'checks': checks,
        'stats_entrada': r['stats_ini'],
    }
    ruta_rep = Path(SALIDA_DIR) / f'{nombre}_report.json'
    with open(ruta_rep, 'w', encoding='utf-8') as f:
        json.dump(informe, f, indent=2, ensure_ascii=False)
    informes.append(informe)

# Copiar a Drive
try:
    Path(SALIDA_DRIVE).mkdir(parents=True, exist_ok=True)
    for f in Path(SALIDA_DIR).glob('*'):
        shutil.copy2(f, Path(SALIDA_DRIVE) / f.name)
    print(f'\nCopiado a Drive: {SALIDA_DRIVE}')
except Exception as e:
    print(f'\n[WARN] Drive: {e}')

# Resumen final
print('\n=== RESUMEN E4 ===')
aptos = sum(1 for i in informes if i['apto_para_imprimir'])
print(f'{aptos}/{len(informes)} mallas aptas para imprimir')
print(f'{"Nombre":<40} {"Apto":>5} {"Water":>6} {"Euler":>6} {"Vol cm3":>8} {"Caras":>7}')
print('-'*72)
for i in informes:
    vol_str = f'{i["volumen_cm3"]:.2f}' if i['volumen_cm3'] else '—'
    print(f'{i["nombre"][:39]:<40} {"SI" if i["apto_para_imprimir"] else "NO":>5} '
          f'{"SI" if i["watertight"] else "NO":>6} {i["euler_number"]:>6} {vol_str:>8} {i["n_caras"]:>7}')

---
## Sección 7 — Visualización 3D final

Renderizado interactivo de las mallas reparadas (rota con el ratón).

In [ ]:
import plotly.graph_objects as go
import numpy as np

_escena = dict(
    xaxis=dict(showticklabels=False, title='', backgroundcolor='#1a1a2e',
               gridcolor='#333', zerolinecolor='#333'),
    yaxis=dict(showticklabels=False, title='', backgroundcolor='#1a1a2e',
               gridcolor='#333', zerolinecolor='#333'),
    zaxis=dict(showticklabels=False, title='', backgroundcolor='#1a1a2e',
               gridcolor='#333', zerolinecolor='#333'),
    bgcolor='#1a1a2e', aspectmode='data'
)

for r, inf in zip(mallas_reparadas, informes):
    mesh   = r['mesh_final']
    nombre = r['nombre']
    verts  = np.array(mesh.vertices)
    faces  = np.array(mesh.faces)

    if len(faces) == 0:
        print(f'[{nombre}] malla vacia'); continue

    # Intensidad por altura (coloring Z)
    intensidad = (verts[:,2] - verts[:,2].min()) / (verts[:,2].ptp() + 1e-8)

    fig = go.Figure(go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        intensity=intensidad,
        colorscale=[[0,'#1a6b8a'],[0.5,'#4ecdc4'],[1,'#e8f4f8']],
        showscale=False,
        lighting=dict(ambient=0.3, diffuse=0.8, roughness=0.4, specular=0.5),
        lightposition=dict(x=200, y=300, z=400)
    ))

    bb = inf['bbox_mm']
    apto_str = 'APTO' if inf['apto_para_imprimir'] else 'REVISAR'
    vol_str  = f'{inf["volumen_cm3"]:.2f} cm3' if inf['volumen_cm3'] else 'N/A'

    fig.update_layout(
        scene=_escena,
        title=dict(
            text=f'<b>{nombre}</b>  [{apto_str}]  '
                 f'{bb[0]:.0f}x{bb[1]:.0f}x{bb[2]:.0f} mm  '
                 f'vol={vol_str}  '
                 f'{len(faces)} caras',
            font=dict(color='white', size=11), x=0.5),
        paper_bgcolor='#1a1a2e',
        font=dict(color='white'),
        height=620, width=700,
        margin=dict(l=0, r=0, t=45, b=0)
    )
    fig.show()

---
## Sección 8 — Comparativa antes / después

Muestra lado a lado el diagnóstico de la malla antes y después de reparar.

In [ ]:
import plotly.graph_objects as go

nombres   = [i['nombre'] for i in informes]
wt_antes  = [i['stats_entrada']['watertight'] for i in informes]
wt_despues= [i['watertight'] for i in informes]
eu_antes  = [i['stats_entrada']['euler'] for i in informes]
eu_despues= [i['euler_number'] for i in informes]
vc_antes  = [i['stats_entrada']['vertices'] for i in informes]
vc_despues= [i['n_vertices'] for i in informes]

fig = go.Figure()
fig.add_trace(go.Bar(name='Vértices antes', x=nombres, y=vc_antes,
                     marker_color='#EF9A9A', opacity=0.8))
fig.add_trace(go.Bar(name='Vértices después', x=nombres, y=vc_despues,
                     marker_color='#A5D6A7', opacity=0.8))
fig.update_layout(
    barmode='group',
    title='Vértices antes vs después de reparar',
    xaxis_tickangle=-30,
    height=400
)
fig.show()

print('\nResumen de reparaciones:')
print(f'{"Nombre":<40} {"Water antes":>11} {"Water desp":>10} {"Euler antes":>11} {"Euler desp":>10}')
print('-'*85)
for i in range(len(informes)):
    print(f'{nombres[i][:39]:<40} {"SI" if wt_antes[i] else "NO":>11} '
          f'{"SI" if wt_despues[i] else "NO":>10} '
          f'{eu_antes[i]:>11} {eu_despues[i]:>10}')